In [ ]:
# Cell 1 — Setup
import pathlib

DPO_RUN_ID = "dpo-YYYYMMDD-XXXX"  # from: alignforge registry list --kind dpo
DRIVE_DIR = "/content/drive/MyDrive/alignforge"

# Verify llama.cpp is built.
if not pathlib.Path("vendor/llama.cpp/build/bin/llama-quantize").exists():
    print("Building llama.cpp — takes ~4 minutes...")
    !bash scripts/setup_llama_cpp.sh
else:
    print("llama.cpp already built.")

In [ ]:
# Cell 2 — Dry run (verify paths before doing anything)
!alignforge export gguf --dpo-run {DPO_RUN_ID} --config configs/export/gguf_q4km.yaml --model-config configs/model/qwen2_5_1_5b.yaml --dry-run

In [ ]:
# Cell 3 — Run the export
# This takes ~10 minutes on CPU (merge 4 min, convert 4 min, quantise 1 min).
!alignforge export gguf --dpo-run {DPO_RUN_ID} --config configs/export/gguf_q4km.yaml --model-config configs/model/qwen2_5_1_5b.yaml --skip-smoke-test
# Ollama not running on Colab — skip smoke test here

In [ ]:
# Cell 4 — Copy GGUF to Drive for safe-keeping
import pathlib
import shutil

gguf_dir = pathlib.Path(f"models/gguf/{DPO_RUN_ID}")
drive_gguf = pathlib.Path(f"{DRIVE_DIR}/models/gguf/{DPO_RUN_ID}")
drive_gguf.mkdir(parents=True, exist_ok=True)

for f in gguf_dir.glob("*.gguf"):
    dest = drive_gguf / f.name
    if not dest.exists():
        print(f"Copying {f.name}...")
        shutil.copy2(f, dest)
    print(f"  ✓ {f.name} → Drive")

# Also copy the Modelfile.
modelfile = gguf_dir / "Modelfile"
if modelfile.exists():
    shutil.copy2(modelfile, drive_gguf / "Modelfile")
    print("  ✓ Modelfile → Drive")

In [ ]:
# Cell 5 — Upload to HuggingFace Hub (optional)
# Requires HF_TOKEN set in .env or environment.
REPO_ID = "<your-hf-username>/alignforge-dpo-1.5b-q4km"

!pip install -q huggingface_hub

from huggingface_hub import HfApi  # noqa: E402

api = HfApi()

# Create the repo if it doesn't exist.
api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True)

# Upload GGUF and Modelfile.
for filename in ["Q4_K_M.gguf", "Modelfile"]:
    local = drive_gguf / filename
    if local.exists():
        api.upload_file(
            path_or_fileobj=str(local),
            path_in_repo=filename,
            repo_id=REPO_ID,
            repo_type="model",
        )
        print(f"Uploaded {filename} to {REPO_ID}")

In [ ]:
# Cell 6 — Verify: inspect the registry
!alignforge registry show {DPO_RUN_ID}
!alignforge registry list